In [ ]:
# !pip install kafka-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import time
import pandas as pd
from kafka import KafkaProducer

#1. Specify the file path in the Jupyter container
csv_filename = "Crimes_-_2026_20260609.csv"
jovyan_path = os.path.join("/home/jovyan/notebooks", csv_filename)

if os.path.exists(csv_filename):
    csv_file_path = csv_filename
elif os.path.exists(jovyan_path):
    csv_file_path = jovyan_path
else:
    csv_file_path = input(f"Please enter full path for {csv_filename}: ")

# 2.Read the data
df = pd.read_csv(csv_file_path)
print(f"✅ Loaded {len(df)} records. Launching Chicago Crime Streamer...")

# 3. Set up the Producer
producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'], 
    value_serializer=lambda x: json.dumps(x).encode('utf-8'),
    api_version=(0, 10),
    acks='all',          
    retries=5            
)

try:
    for index, row in df.iterrows():
        message = row.to_dict()
        
        # Filter out NaN values to ensure compatibility with JSON
        for key, value in message.items():
            if pd.isna(value): 
                message[key] = None

        # Clean core text fields for filters and Windows in Flink
        string_cols = ['Primary Type', 'Description', 'Location Description', 'Block']
        for col in string_cols:
            if isinstance(message.get(col), str):
                message[col] = message[col].strip()

        # Set flags as explicit Boolean values
        # Pandas sometimes reads true/false as strings, so we convert them to actual Boolean values
        message['Arrest'] = str(message.get('Arrest')).lower() == 'true'
        message['Domestic'] = str(message.get('Domestic')).lower() == 'true'
        
        # Adjust administrative identifiers (District & Beat) to be proper integers
        try:
            message['District'] = int(float(message['District'])) if message['District'] is not None else 0
            message['Beat'] = int(float(message['Beat'])) if message['Beat'] is not None else 0
        except (ValueError, TypeError):
            message['District'] = 0
            message['Beat'] = 0

        # Define the Kafka Key based on District to ensure crimes from the same district are routed to the same partition
        kafka_key = str(message['District']).encode('utf-8')
        
        # Stream the message to the new Topic
        producer.send('chicago_crimes_stream', key=kafka_key, value=message)
        
        if index % 100 == 0:
            print(f"[STREAMING] Dispatched record #{index} | ID: {message['ID']} | Type: {message['Primary Type']} | District: {message['District']}")
            
        # Control streaming speed (0.05 seconds per record to simulate a suitable pace for observing real-time Windows)
        time.sleep(0.05) 

except KeyboardInterrupt:
    print("\n[INFO] Stream paused by user.")
finally:
    producer.flush()
    producer.close()
    print("[INFO] Producer closed safely.")

✅ Loaded 90948 records. Launching Chicago Crime Streamer...
[INFO] Producer closed safely.


KafkaTimeoutError: KafkaTimeoutError: Failed to update metadata after 60.0 secs.